In [1]:
!pip install backtesting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 7.7 MB/s eta 0:00:00


In [2]:
pip install pandas_ta

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.1/115.1 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pandas_ta: filename=pandas_ta-0.3.14b0-py3-none-any.whl size=218910 sha256=a877ca47d1ff0511a481e24d767eaed30aec556ae505de11d2882646d8119eeb
  Stored in directory: /root/.cache/pip/wheels/7f/33/8b/50b245c5c65433cd8f5cb24ac15d97e5a3db2d41a8b6ae957d
Successfully built pandas_ta


In [4]:
import pandas_ta as ta

In [5]:
from datetime import datetime
import requests
import pandas as pd
import numpy as np
import time
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

/usr/local/lib/python3.11/dist-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


In [6]:

# --- HÀM TẢI DỮ LIỆU TỪ BINANCE ---
def str_to_millis(date_str):
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    return int(dt.timestamp() * 1000)

def fetch_binance_kline_df(symbol, interval, start_str, end_str):
    base_url = "https://api.binance.us/"
    api_url = "api/v3/klines"
    general_url = base_url + api_url

    start_time = str_to_millis(start_str)
    end_dt = datetime.strptime(end_str, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
    end_time = int(end_dt.timestamp() * 1000)

    all_klines = []
    while start_time < end_time:
        params = {
            "symbol": symbol.upper(),
            "interval": interval,
            "startTime": start_time,
            "endTime": end_time,
            "limit": 1000
        }
        response = requests.get(general_url, params=params)
        if response.status_code != 200:
            print("Error:", response.status_code, response.text)
            break

        data = response.json()
        if not data:
            break
        all_klines.extend(data)

        last_ts = data[-1][0]
        start_time = last_ts + 1
        time.sleep(0.3)

    df = pd.DataFrame(all_klines, columns=[
        "timestamp", "o", "h", "l", "c", "v",
        "close_time", "quote_asset_volume", "num_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"
    ])
    df = df[["timestamp", "o", "c", "h", "l", "v"]]
    df.columns = ["timestamp", "Open", "High", "Low", "Close", "Volume"]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df.set_index("timestamp", inplace=True)
    df = df.astype(float)
    return df

In [15]:
# --- NHẬP THỜI GIAN ---
symbol = "BTCUSDT"
interval = "1m"
start_date = "01/01/2020"
end_date = "31/12/2025"

print("\U0001F4E5 Đang tải dữ liệu từ Binance...")
df_all = fetch_binance_kline_df(symbol, interval, start_date, end_date)
# print(df_all)
print(f"✅ Dữ liệu đã tải: {len(df_all)} dòng, từ {start_date} đến {end_date}")


📥 Đang tải dữ liệu từ Binance...
✅ Dữ liệu đã tải: 2959925 dòng, từ 01/01/2020 đến 31/12/2025


In [16]:
class MAStrategy_noRSI(Strategy):
    short_ma = 2
    long_ma = 5

    def init(self):
        close = pd.Series(self.data.Close)

        # Tính SMA
        self.ma_short = self.I(lambda x: pd.Series(x).rolling(self.short_ma).mean(), self.data.Close)
        self.ma_long = self.I(lambda x: pd.Series(x).rolling(self.long_ma).mean(), self.data.Close)

        # Tính RSI bằng pandas_ta
        self.rsi = self.I(lambda x: ta.rsi(pd.Series(x), length=5), self.data.Close)

    def next(self):
        price = self.data.Close[-1]
        ma_s = self.ma_short[-1]
        ma_l = self.ma_long[-1]
        rsi_val = self.rsi[-1]

        # Vào lệnh Long
        # if ma_s > ma_l and rsi_val >58:
        #     if not self.position.is_long:
        #         self.position.close()
        #         self.buy(sl=price*0.95, tp=price*1.05,size=0.9)
        if ma_s > ma_l :
            if not self.position.is_long:
                self.position.close()
                self.buy(sl=price*0.95, tp=price*1.05,size=100)

        # # # # Vào lệnh Short
        # if ma_s < ma_l and rsi_val < 37:
        #     if not self.position.is_short:
        #         self.position.close()
        #         self.sell(sl=price*1.005, tp=price*0.995,size=1)

In [17]:
bt = Backtest(df_all, MAStrategy_noRSI,
              cash=10000000,
              commission=0.001,
              trade_on_close=True)

stats = bt.run()
bt.plot()
print(stats)

Backtest.run:   0%|          | 0/2959917 [00:00<?, ?bar/s]

/tmp/ipython-input-1632872808.py:6: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()
/usr/local/lib/python3.11/dist-packages/backtesting/_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


Start                     2020-01-01 00:00:00
End                       2025-08-19 08:23:00
Duration                   2057 days 08:23:00
Exposure Time [%]                    99.51556
Equity Final [$]                21581853.9084
Equity Peak [$]                 22514886.9084
Commissions [$]                  6395859.9506
Return [%]                          115.81854
Buy & Hold Return [%]               1499.5978
Return (Ann.) [%]                     14.6165
Volatility (Ann.) [%]                19.79121
CAGR [%]                             14.62296
Sharpe Ratio                          0.73854
Sortino Ratio                         1.27346
Calmar Ratio                          0.50915
Alpha [%]                          -244.79842
Beta                                  0.24048
Max. Drawdown [%]                   -28.70742
Avg. Drawdown [%]                    -0.26324
Max. Drawdown Duration      840 days 02:58:00
Avg. Drawdown Duration        1 days 09:44:00
# Trades                          

In [18]:
class MAStrategy(Strategy):
    short_ma = 5
    long_ma = 10

    def init(self):
        close = pd.Series(self.data.Close)

        # Tính SMA
        self.ma_short = self.I(lambda x: pd.Series(x).rolling(self.short_ma).mean(), self.data.Close)
        self.ma_long = self.I(lambda x: pd.Series(x).rolling(self.long_ma).mean(), self.data.Close)

        # Tính RSI bằng pandas_ta
        self.rsi = self.I(lambda x: ta.rsi(pd.Series(x), length=5), self.data.Close)

    def next(self):
        price = self.data.Close[-1]
        ma_s = self.ma_short[-1]
        ma_l = self.ma_long[-1]
        rsi_val = self.rsi[-1]

        # Vào lệnh Long
        if ma_s > ma_l and rsi_val >58:
            if not self.position.is_long:
                self.position.close()
                self.buy(sl=price*0.95, tp=price*1.05,size=0.9)
        # if ma_s > ma_l :
        #     if not self.position.is_long:
        #         self.position.close()
        #         self.buy(sl=price*0.95, tp=price*1.05,size=0.9)

        # # # # Vào lệnh Short
        # if ma_s < ma_l and rsi_val < 37:
        #     if not self.position.is_short:
        #         self.position.close()
        #         self.sell(sl=price*1.005, tp=price*0.995,size=1)

In [19]:
bt = Backtest(df_all, MAStrategy,
              cash=10000000,
              commission=0.001,
              trade_on_close=True)

stats = bt.run()
bt.plot()
print(stats)

Backtest.run:   0%|          | 0/2959915 [00:00<?, ?bar/s]

/tmp/ipython-input-3098581266.py:6: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()
/usr/local/lib/python3.11/dist-packages/backtesting/_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


Start                     2020-01-01 00:00:00
End                       2025-08-19 08:23:00
Duration                   2057 days 08:23:00
Exposure Time [%]                    99.39735
Equity Final [$]              176791418.15059
Equity Peak [$]               189667273.55059
Commissions [$]                96352480.83081
Return [%]                         1667.91418
Buy & Hold Return [%]              1499.46415
Return (Ann.) [%]                    66.40007
Volatility (Ann.) [%]               105.86746
CAGR [%]                             66.46216
Sharpe Ratio                           0.6272
Sortino Ratio                         1.78034
Calmar Ratio                          0.90411
Alpha [%]                           387.48119
Beta                                  0.85393
Max. Drawdown [%]                   -73.44283
Avg. Drawdown [%]                     -0.8899
Max. Drawdown Duration      845 days 03:09:00
Avg. Drawdown Duration        1 days 09:07:00
# Trades                          